In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display
import gradio as gr

In [ ]:
load_dotenv(override=True)
gemini_api_key = os.getenv('GEMINI_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY2')
mistral_api_key = os.getenv('MISTRAL_API_KEY')
moonshotai_api_key = os.getenv('MOONSHOTAI_API_KEY')
qwen_api_key=os.getenv('QWEN_API_KEY')


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
NVIDIA_BASE_URL= "https://integrate.api.nvidia.com/v1"

gemini=OpenAI(base_url=GEMINI_BASE_URL,api_key=gemini_api_key)
deepseek = OpenAI(base_url=NVIDIA_BASE_URL,api_key=deepseek_api_key)
mistral = OpenAI(base_url=NVIDIA_BASE_URL,api_key=mistral_api_key)
moonshot = OpenAI(base_url=NVIDIA_BASE_URL,api_key=moonshotai_api_key)
qwen = OpenAI(base_url=NVIDIA_BASE_URL,api_key=qwen_api_key)





In [ ]:
GEMINI_MODEL="gemini-3.1-pro-preview"
QWEN_MODEL="qwen/qwen3-coder-480b-a35b-instruct"
MISTRAL_MODEL="mistralai/mistral-medium-3.5-128b"
KIMI_MODEL="moonshotai/kimi-k2.6"

In [ ]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

In [ ]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = gemini.chat.completions.create(model=GEMINI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

In [ ]:
compile_command = [
    "clang++",
    "-Ofast",
    "-march=native",
    "-flto",
    "-DNDEBUG",
    "-std=c++20",
    "main.cpp",
    "-o",
    "main"
]

run_command = ["./main"]

In [ ]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time and you can also make use of threading :).
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [ ]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [ ]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [ ]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [ ]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [ ]:
run_python(pi)

In [ ]:
port(gemini, "gemini-2.5-pro", pi)

In [ ]:
import subprocess

def compile_and_run(runs=3):
    try:
        # Compile once
        subprocess.run(
            compile_command,
            check=True,
            text=True,
            capture_output=True
        )

        # Run multiple times
        for i in range(runs):
            result = subprocess.run(
                run_command,
                check=True,
                text=True,
                capture_output=True
            )

            print(f"Run {i+1}:")
            print(result.stdout)

    except subprocess.CalledProcessError as e:
        print("Error:\n")

        if e.stderr:
            print(e.stderr)
        else:
            print(e)

In [ ]:
compile_and_run()

In [ ]:
port(qwen, QWEN_MODEL, pi)
compile_and_run()

In [ ]:
port(mistral, MISTRAL_MODEL, pi)
compile_and_run()

In [ ]:
port(moonshot, KIMI_MODEL, pi)
compile_and_run()

In [ ]:
models=["gemini-2.5-pro",MISTRAL_MODEL,QWEN_MODEL,KIMI_MODEL]
clients = {
    "gemini-2.5-pro": gemini,
    "qwen/qwen3-coder-480b-a35b-instruct": qwen,
    "mistralai/mistral-medium-3.5-128b": mistral,
    "moonshotai/kimi-k2.6": moonshot
}

In [ ]:
def port2(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
    return reply

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port2, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)